# Evidence-Based Personal Hybrid Insider-Threat Detection

This version keeps the original three-autoencoder design. It makes
three changes supported by the papers reviewed:

1. every stream includes lagged personal-history deviations, so a user is
   compared with their own recent behaviour;
2. stream evidence is summarized by agreement rather than allowing a maximum
   score to be treated as sufficient evidence;
3. a small regularized logistic model uses official pre-test labels to separate
   technical unusualness from suspected malicious behaviour.



This version keeps the original objective and model unchanged:

- one autoencoder for logon behaviour;
- one autoencoder for device behaviour;
- one autoencoder for file behaviour;
- the same daily features and 32–16–8–16–32 architecture;
- chronological 70% training, 15% validation and 15% testing;
- stream reconstruction errors converted to validation percentiles;
- missing stream scores remain missing;
- OCEAN traits are added only after technical anomaly detection;
- official CERT user-day labels are used for evaluation.

Only two focused improvements are added. The validation data selects whether the ensemble should use the **mean** or **maximum** available stream percentile, and it selects an ensemble threshold from a small documented list. The final test labels are not used for either choice.

In [1]:
import os

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn

from google.colab import drive


SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

WORK_START = 8
WORK_END = 18

TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15

STREAM_THRESHOLD_PERCENTILE = 0.95
# The original 0.95 value remains a candidate, but validation labels now
# compare it with stricter thresholds instead of selecting it blindly.
ENSEMBLE_THRESHOLD_CANDIDATES = [0.95, 0.97, 0.98, 0.99, 0.995, 0.999]

OCEAN_LOW_PERCENTILE = 0.05
OCEAN_HIGH_PERCENTILE = 0.95

EPOCHS = 50
PATIENCE = 7
LEARNING_RATE = 0.001
BATCH_SIZE = 2048

DATE_FORMAT = "%m/%d/%Y %H:%M:%S"

print("Training device:", DEVICE)


Training device: cpu


## Mounting Google Drive and locate the CERT files

In [2]:
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive"

required_names = {
    "logon.csv",
    "device.csv",
    "file.csv",
    "psychometric.csv"
}

matching_folders = []

for current_folder, subfolders, filenames in os.walk(DRIVE_ROOT):
    lowercase_names = {
        filename.lower()
        for filename in filenames
    }

    if required_names.issubset(lowercase_names):
        matching_folders.append(current_folder)

if len(matching_folders) == 0:
    raise FileNotFoundError(
        "No My Drive folder contains logon.csv, device.csv, "
        "file.csv and psychometric.csv together. If the folder "
        "is shared, add a shortcut to My Drive and run again."
    )

if len(matching_folders) > 1:
    print("More than one matching folder was found:")

    for folder in matching_folders:
        print(folder)

    raise ValueError(
        "Set DATA_FOLDER manually to the correct folder shown above."
    )

DATA_FOLDER = matching_folders[0]
OUTPUT_FOLDER = os.path.join(
    DATA_FOLDER,
    "CERT_model_outputs"
)

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)


def find_filename(folder, expected_name):
    for filename in os.listdir(folder):
        if filename.lower() == expected_name.lower():
            return os.path.join(folder, filename)

    raise FileNotFoundError(expected_name)


LOGON_PATH = find_filename(DATA_FOLDER, "logon.csv")
DEVICE_PATH = find_filename(DATA_FOLDER, "device.csv")
FILE_PATH = find_filename(DATA_FOLDER, "file.csv")
PSYCHOMETRIC_PATH = find_filename(
    DATA_FOLDER,
    "psychometric.csv"
)

required_files = {
    "logon.csv": LOGON_PATH,
    "device.csv": DEVICE_PATH,
    "file.csv": FILE_PATH,
    "psychometric.csv": PSYCHOMETRIC_PATH
}

print("Data folder:", DATA_FOLDER)
print("Output folder:", OUTPUT_FOLDER)

for filename, path in required_files.items():
    size_in_mb = os.path.getsize(path) / (1024 ** 2)

    print(
        filename,
        "-",
        round(size_in_mb, 2),
        "MB"
    )


Mounted at /content/drive
Data folder: /content/drive/MyDrive/r6.2
Output folder: /content/drive/MyDrive/r6.2/CERT_model_outputs
logon.csv - 230.45 MB
device.csv - 133.08 MB
file.csv - 1269.62 MB
psychometric.csv - 0.17 MB


## Load Selected CSV Columns

In [3]:
def load_columns(path, required_columns):
    header = pd.read_csv(path, nrows=0)

    column_lookup = {
        column.strip().lower(): column
        for column in header.columns
    }

    missing_columns = [
        column
        for column in required_columns
        if column not in column_lookup
    ]

    if missing_columns:
        raise ValueError(
            f"{os.path.basename(path)} is missing: {missing_columns}. "
            f"Available columns: {list(header.columns)}"
        )

    original_column_names = [
        column_lookup[column]
        for column in required_columns
    ]

    dataframe = pd.read_csv(
        path,
        usecols=original_column_names,
        low_memory=False
    )

    dataframe.columns = [
        column.strip().lower()
        for column in dataframe.columns
    ]

    return dataframe

## Prepare timestamps and common event fields

In [4]:
def prepare_events(dataframe):
    dataframe = dataframe.copy()

    parsed_dates = pd.to_datetime(
        dataframe["date"],
        format=DATE_FORMAT,
        errors="coerce"
    )

    if parsed_dates.isna().mean() > 0.01:
        parsed_dates = pd.to_datetime(
            dataframe["date"],
            errors="coerce"
        )

    dataframe["date"] = parsed_dates

    dataframe = dataframe.dropna(
        subset=["date", "user", "pc"]
    ).copy()

    dataframe["user"] = (
        dataframe["user"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    dataframe["pc"] = (
        dataframe["pc"]
        .astype(str)
        .str.strip()
    )

    dataframe["day"] = dataframe["date"].dt.normalize()
    dataframe["hour"] = dataframe["date"].dt.hour
    dataframe["weekday"] = dataframe["date"].dt.weekday

    dataframe["weekend"] = (
        dataframe["weekday"] >= 5
    ).astype(int)

    dataframe["after_hours"] = (
        (dataframe["hour"] < WORK_START)
        | (dataframe["hour"] >= WORK_END)
    ).astype(int)

    return dataframe

## Create logon features

In [5]:
def build_logon_features(dataframe):
    dataframe = prepare_events(dataframe)

    dataframe["activity"] = (
        dataframe["activity"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    dataframe["is_logon"] = (
        dataframe["activity"] == "logon"
    ).astype(int)

    dataframe["is_logoff"] = (
        dataframe["activity"] == "logoff"
    ).astype(int)

    daily = dataframe.groupby(
        ["user", "day"],
        as_index=False
    ).agg(
        logon_total=("activity", "size"),
        logon_count=("is_logon", "sum"),
        logoff_count=("is_logoff", "sum"),
        logon_unique_pcs=("pc", "nunique"),
        logon_first_hour=("hour", "min"),
        logon_last_hour=("hour", "max"),
        logon_after_hours=("after_hours", "sum"),
        logon_weekend=("weekend", "max"),
        logon_hour_std=("hour", "std")
    )

    daily["logon_hour_std"] = (
        daily["logon_hour_std"].fillna(0)
    )

    daily["logon_after_hours_ratio"] = (
        daily["logon_after_hours"]
        / daily["logon_total"]
    )

    daily["logon_logoff_ratio"] = (
        daily["logon_count"]
        / (daily["logoff_count"] + 1)
    )

    return daily

## Creating device features

In [6]:
def build_device_features(dataframe):
    dataframe = prepare_events(dataframe)

    dataframe["activity"] = (
        dataframe["activity"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    dataframe["is_connect"] = (
        dataframe["activity"] == "connect"
    ).astype(int)

    dataframe["is_disconnect"] = (
        dataframe["activity"] == "disconnect"
    ).astype(int)

    daily = dataframe.groupby(
        ["user", "day"],
        as_index=False
    ).agg(
        device_total=("activity", "size"),
        connect_count=("is_connect", "sum"),
        disconnect_count=("is_disconnect", "sum"),
        device_unique_pcs=("pc", "nunique"),
        device_first_hour=("hour", "min"),
        device_last_hour=("hour", "max"),
        device_after_hours=("after_hours", "sum"),
        device_weekend=("weekend", "max"),
        device_hour_std=("hour", "std")
    )

    daily["device_hour_std"] = (
        daily["device_hour_std"].fillna(0)
    )

    daily["device_after_hours_ratio"] = (
        daily["device_after_hours"]
        / daily["device_total"]
    )

    daily["connect_disconnect_ratio"] = (
        daily["connect_count"]
        / (daily["disconnect_count"] + 1)
    )

    return daily

## Creating file features

In [7]:
def build_file_features(dataframe):
    dataframe = prepare_events(dataframe)

    dataframe["filename"] = (
        dataframe["filename"]
        .astype(str)
        .str.strip()
    )

    daily = dataframe.groupby(
        ["user", "day"],
        as_index=False
    ).agg(
        file_total=("filename", "size"),
        unique_files=("filename", "nunique"),
        file_unique_pcs=("pc", "nunique"),
        file_first_hour=("hour", "min"),
        file_last_hour=("hour", "max"),
        file_after_hours=("after_hours", "sum"),
        file_weekend=("weekend", "max"),
        file_hour_std=("hour", "std")
    )

    daily["file_hour_std"] = (
        daily["file_hour_std"].fillna(0)
    )

    daily["file_after_hours_ratio"] = (
        daily["file_after_hours"]
        / daily["file_total"]
    )

    return daily

## Loading and aggregrating the three datasets

In [8]:
logon_raw = load_columns(
    LOGON_PATH,
    ["date", "user", "pc", "activity"]
)

logon_features = build_logon_features(logon_raw)
del logon_raw

device_raw = load_columns(
    DEVICE_PATH,
    ["date", "user", "pc", "activity"]
)

device_features = build_device_features(device_raw)
del device_raw

file_raw = load_columns(
    FILE_PATH,
    ["date", "user", "pc", "filename"]
)

file_features = build_file_features(file_raw)
del file_raw

print("Logon profiles:", len(logon_features))
print("Device profiles:", len(device_features))
print("File profiles:", len(file_features))

Logon profiles: 1394010
Device profiles: 198993
File profiles: 308647


In [9]:
PERSONAL_WINDOW_DAYS = 30
PERSONAL_MIN_HISTORY = 5


def add_personal_history_features(dataframe, base_features):
    """Add leakage-safe deviations using only each user's earlier profiles."""
    result = dataframe.sort_values(["user", "day"]).copy()

    for feature in base_features:
        previous_values = result.groupby("user")[feature].shift(1)

        rolling_mean = (
            previous_values
            .groupby(result["user"])
            .transform(
                lambda values: values.rolling(
                    PERSONAL_WINDOW_DAYS,
                    min_periods=PERSONAL_MIN_HISTORY
                ).mean()
            )
        )

        rolling_std = (
            previous_values
            .groupby(result["user"])
            .transform(
                lambda values: values.rolling(
                    PERSONAL_WINDOW_DAYS,
                    min_periods=PERSONAL_MIN_HISTORY
                ).std()
            )
        )

        personal_z = (
            (result[feature] - rolling_mean)
            / (rolling_std + 1e-6)
        )

        # Zero means "insufficient evidence of personal deviation". Clipping
        # prevents a nearly-zero historical standard deviation from dominating.
        result[feature + "_personal_z"] = (
            personal_z
            .replace([np.inf, -np.inf], np.nan)
            .clip(-10, 10)
            .fillna(0.0)
        )

    return result


## Define the model features

These lists were missing from the attached notebook. They explicitly identify which engineered columns enter each autoencoder.


In [10]:
LOGON_BASE_FEATURES = [
    "logon_total", "logon_count", "logoff_count", "logon_unique_pcs",
    "logon_first_hour", "logon_last_hour", "logon_after_hours",
    "logon_weekend", "logon_hour_std", "logon_after_hours_ratio",
    "logon_logoff_ratio"
]

DEVICE_BASE_FEATURES = [
    "device_total", "connect_count", "disconnect_count", "device_unique_pcs",
    "device_first_hour", "device_last_hour", "device_after_hours",
    "device_weekend", "device_hour_std", "device_after_hours_ratio",
    "connect_disconnect_ratio"
]

FILE_BASE_FEATURES = [
    "file_total", "unique_files", "file_unique_pcs", "file_first_hour",
    "file_last_hour", "file_after_hours", "file_weekend", "file_hour_std",
    "file_after_hours_ratio"
]

logon_features = add_personal_history_features(
    logon_features, LOGON_BASE_FEATURES
)
device_features = add_personal_history_features(
    device_features, DEVICE_BASE_FEATURES
)
file_features = add_personal_history_features(
    file_features, FILE_BASE_FEATURES
)

LOGON_FEATURES = LOGON_BASE_FEATURES + [
    feature + "_personal_z" for feature in LOGON_BASE_FEATURES
]
DEVICE_FEATURES = DEVICE_BASE_FEATURES + [
    feature + "_personal_z" for feature in DEVICE_BASE_FEATURES
]
FILE_FEATURES = FILE_BASE_FEATURES + [
    feature + "_personal_z" for feature in FILE_BASE_FEATURES
]

print("Logon model features:", len(LOGON_FEATURES))
print("Device model features:", len(DEVICE_FEATURES))
print("File model features:", len(FILE_FEATURES))


Logon model features: 22
Device model features: 22
File model features: 18


## Defining model features and chronological dates

In [11]:
logon_features["day"] = pd.to_datetime(logon_features["day"])
device_features["day"] = pd.to_datetime(device_features["day"])
file_features["day"] = pd.to_datetime(file_features["day"])


print(
    "Logon date range:",
    logon_features["day"].min(),
    "to",
    logon_features["day"].max()
)

print(
    "Device date range:",
    device_features["day"].min(),
    "to",
    device_features["day"].max()
)

print(
    "File date range:",
    file_features["day"].min(),
    "to",
    file_features["day"].max()
)


COMMON_START_DAY = max(
    logon_features["day"].min(),
    device_features["day"].min(),
    file_features["day"].min()
)


COMMON_END_DAY = min(
    logon_features["day"].max(),
    device_features["day"].max(),
    file_features["day"].max()
)


if COMMON_START_DAY >= COMMON_END_DAY:
    raise ValueError(
        "The logon, device and file datasets do not have "
        "an overlapping date range. Check that all files "
        "come from the same CERT release."
    )


print(
    "\nCommon date range:",
    COMMON_START_DAY,
    "to",
    COMMON_END_DAY
)


logon_features = logon_features[
    (logon_features["day"] >= COMMON_START_DAY)
    & (logon_features["day"] <= COMMON_END_DAY)
].copy()


device_features = device_features[
    (device_features["day"] >= COMMON_START_DAY)
    & (device_features["day"] <= COMMON_END_DAY)
].copy()


file_features = file_features[
    (file_features["day"] >= COMMON_START_DAY)
    & (file_features["day"] <= COMMON_END_DAY)
].copy()


common_days = pd.date_range(
    start=COMMON_START_DAY,
    end=COMMON_END_DAY,
    freq="D"
)


if len(common_days) < 20:
    raise ValueError(
        "There are too few shared dates for a reliable "
        "training-validation-test split."
    )


train_end_position = int(
    len(common_days) * TRAIN_RATIO
)


validation_end_position = int(
    len(common_days)
    * (TRAIN_RATIO + VALIDATION_RATIO)
)


train_end_position = max(
    1,
    min(train_end_position, len(common_days) - 2)
)

validation_end_position = max(
    train_end_position + 1,
    min(validation_end_position, len(common_days) - 1)
)


TRAIN_END_DAY = common_days[train_end_position]


VALIDATION_END_DAY = common_days[
    validation_end_position
]


print("\nTraining period:")
print(COMMON_START_DAY, "to", TRAIN_END_DAY - pd.Timedelta(days=1))

print("\nValidation period:")
print(
    TRAIN_END_DAY,
    "to",
    VALIDATION_END_DAY - pd.Timedelta(days=1)
)

print("\nTest period:")
print(VALIDATION_END_DAY, "to", COMMON_END_DAY)

Logon date range: 2010-01-02 00:00:00 to 2011-06-01 00:00:00
Device date range: 2010-01-02 00:00:00 to 2011-05-31 00:00:00
File date range: 2010-01-02 00:00:00 to 2011-05-31 00:00:00

Common date range: 2010-01-02 00:00:00 to 2011-05-31 00:00:00

Training period:
2010-01-02 00:00:00 to 2010-12-27 00:00:00

Validation period:
2010-12-28 00:00:00 to 2011-03-14 00:00:00

Test period:
2011-03-15 00:00:00 to 2011-05-31 00:00:00


## split check

In [12]:
def check_split_sizes(dataframe, dataset_name):
    train_rows = dataframe[
        dataframe["day"] < TRAIN_END_DAY
    ]

    validation_rows = dataframe[
        (dataframe["day"] >= TRAIN_END_DAY)
        & (dataframe["day"] < VALIDATION_END_DAY)
    ]

    test_rows = dataframe[
        dataframe["day"] >= VALIDATION_END_DAY
    ]

    print(f"\n{dataset_name}")
    print("Training rows:", len(train_rows))
    print("Validation rows:", len(validation_rows))
    print("Test rows:", len(test_rows))

    if train_rows.empty:
        raise ValueError(
            f"{dataset_name} training split is empty."
        )

    if validation_rows.empty:
        raise ValueError(
            f"{dataset_name} validation split is empty."
        )

    if test_rows.empty:
        raise ValueError(
            f"{dataset_name} test split is empty."
        )


check_split_sizes(
    logon_features,
    "Logon dataset"
)

check_split_sizes(
    device_features,
    "Device dataset"
)

check_split_sizes(
    file_features,
    "File dataset"
)


Logon dataset
Training rows: 987021
Validation rows: 205061
Test rows: 201915

Device dataset
Training rows: 141472
Validation rows: 29017
Test rows: 28504

File dataset
Training rows: 219713
Validation rows: 44257
Test rows: 44677


## Extract official ground truth for validation calibration

The autoencoders remain unsupervised. Labels are introduced only after the chronological split has been fixed. Validation labels select the ensemble rule and threshold; test labels remain untouched until final evaluation.

In [13]:
import glob
import csv
import io
import tarfile

CERT_RELEASE = "6.2"
ANSWER_ARCHIVE_PATH = os.path.join(DATA_FOLDER, "answers.tar.bz2")

if not os.path.exists(ANSWER_ARCHIVE_PATH):
    candidates = glob.glob(
        DRIVE_ROOT + "/**/answers.tar.bz2",
        recursive=True
    )

    if len(candidates) != 1:
        raise FileNotFoundError(
            "Exactly one answers.tar.bz2 archive must be available."
        )

    ANSWER_ARCHIVE_PATH = candidates[0]

ground_truth_records = []

with tarfile.open(ANSWER_ARCHIVE_PATH, "r:bz2") as archive:
    insiders_file = archive.extractfile("answers/insiders.csv")

    if insiders_file is None:
        raise FileNotFoundError("answers/insiders.csv is missing.")

    insiders = pd.read_csv(insiders_file, dtype=str)
    incidents = insiders[insiders["dataset"] == CERT_RELEASE]

    for _, incident in incidents.iterrows():
        insider_user = incident["user"].strip().upper()
        details_path = "answers/" + incident["details"]
        details_file = archive.extractfile(details_path)

        if details_file is None:
            raise FileNotFoundError(details_path)

        text_file = io.TextIOWrapper(
            details_file,
            encoding="utf-8",
            errors="replace",
            newline=""
        )

        for row in csv.reader(text_file):
            if len(row) < 4:
                continue

            event_date = pd.to_datetime(
                row[2],
                format=DATE_FORMAT,
                errors="coerce"
            )
            event_user = row[3].strip().upper()

            if pd.notna(event_date) and event_user == insider_user:
                ground_truth_records.append({
                    "user": insider_user,
                    "day": event_date.normalize(),
                    "scenario_id": (
                        "r" + CERT_RELEASE
                        + "-scenario-"
                        + str(incident["scenario"])
                    )
                })

ground_truth = (
    pd.DataFrame(ground_truth_records)
    .drop_duplicates(["user", "day", "scenario_id"])
    .sort_values(["day", "user"])
    .reset_index(drop=True)
)

if ground_truth.empty:
    raise ValueError("No malicious user-days were extracted.")

daily_ground_truth = (
    ground_truth
    .groupby(["user", "day"], as_index=False)
    .agg(
        scenario_id=(
            "scenario_id",
            lambda values: " | ".join(sorted(set(values)))
        )
    )
)

daily_ground_truth["ground_truth_label"] = 1

GROUND_TRUTH_PATH = os.path.join(DATA_FOLDER, "ground_truth.csv")
ground_truth.to_csv(GROUND_TRUTH_PATH, index=False)

ground_truth["split"] = np.select(
    [
        ground_truth["day"] < TRAIN_END_DAY,
        ground_truth["day"] < VALIDATION_END_DAY
    ],
    ["Training", "Validation"],
    default="Test"
)

print("Total malicious user-days:", len(daily_ground_truth))
print("Ground truth saved to:", GROUND_TRUTH_PATH)
print(ground_truth.groupby("split").size())

Total malicious user-days: 44
Ground truth saved to: /content/drive/MyDrive/r6.2/ground_truth.csv
split
Test          12
Training       6
Validation    26
dtype: int64


## Building the autoencoder

In [14]:
class Autoencoder(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, input_size)
        )

    def forward(self, values):
        compressed = self.encoder(values)
        reconstructed = self.decoder(compressed)

        return reconstructed

## Calculating the reconstruction error

In [15]:
def reconstruction_errors(model, values):
    errors = []

    model.eval()

    with torch.no_grad():
        for start in range(0, len(values), BATCH_SIZE):
            batch = values[
                start:start + BATCH_SIZE
            ].to(DEVICE)

            reconstructed = model(batch)

            batch_errors = (
                (batch - reconstructed) ** 2
            ).mean(dim=1)

            errors.extend(
                batch_errors.cpu().numpy()
            )

    return np.array(errors)

## Converting raw scores into comparable percentiles

In [16]:
def percentile_scores(reference_errors, new_errors):
    sorted_reference = np.sort(reference_errors)

    scores = np.searchsorted(
        sorted_reference,
        new_errors,
        side="right"
    ) / len(sorted_reference)

    return scores

## Training one autoencoder correctly

In [17]:
def train_stream(dataframe, feature_columns, stream_name):
    dataframe = dataframe.sort_values(
        ["day", "user"]
    ).reset_index(drop=True)

    train_data = dataframe[
        dataframe["day"] < TRAIN_END_DAY
    ].copy()

    validation_data = dataframe[
        (dataframe["day"] >= TRAIN_END_DAY)
        & (dataframe["day"] < VALIDATION_END_DAY)
    ].copy()

    test_data = dataframe[
        dataframe["day"] >= VALIDATION_END_DAY
    ].copy()

    # Controlled normal-baseline experiment: official malicious labels are
    # removed from autoencoder training only. Validation and test are untouched.
    malicious_training_days = daily_ground_truth[["user", "day"]].copy()
    train_data = (
        train_data
        .merge(
            malicious_training_days.assign(_known_malicious=1),
            on=["user", "day"],
            how="left"
        )
    )
    removed_training_rows = int(train_data["_known_malicious"].fillna(0).sum())
    train_data = train_data[
        train_data["_known_malicious"].isna()
    ].drop(columns="_known_malicious")
    print("Known malicious training profiles removed:", removed_training_rows)

    print(f"\nTraining {stream_name} autoencoder")
    print("Training rows:", len(train_data))
    print("Validation rows:", len(validation_data))
    print("Test rows:", len(test_data))

    if train_data.empty:
        raise ValueError(
            f"{stream_name} has no training rows. "
            "Check its date range."
        )

    if validation_data.empty:
        raise ValueError(
            f"{stream_name} has no validation rows. "
            "Check TRAIN_END_DAY and VALIDATION_END_DAY."
        )

    if test_data.empty:
        raise ValueError(
            f"{stream_name} has no test rows. "
            "Check the dataset's final date."
        )

    scaler = StandardScaler()

    train_scaled = scaler.fit_transform(
        train_data[feature_columns]
    )

    validation_scaled = scaler.transform(
        validation_data[feature_columns]
    )

    test_scaled = scaler.transform(
        test_data[feature_columns]
    )

    train_values = torch.tensor(
        train_scaled,
        dtype=torch.float32
    )

    validation_values = torch.tensor(
        validation_scaled,
        dtype=torch.float32
    )

    test_values = torch.tensor(
        test_scaled,
        dtype=torch.float32
    )

    torch.manual_seed(SEED)

    model = Autoencoder(
        len(feature_columns)
    ).to(DEVICE)

    loss_function = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    best_validation_loss = np.inf
    best_weights = None
    waiting_epochs = 0

    for epoch in range(EPOCHS):
        model.train()

        row_order = torch.randperm(
            len(train_values)
        )

        total_training_loss = 0

        for start in range(
            0,
            len(train_values),
            BATCH_SIZE
        ):
            row_numbers = row_order[
                start:start + BATCH_SIZE
            ]

            batch = train_values[
                row_numbers
            ].to(DEVICE)

            optimizer.zero_grad()

            reconstructed = model(batch)

            loss = loss_function(
                reconstructed,
                batch
            )

            loss.backward()
            optimizer.step()

            total_training_loss += (
                loss.item() * len(batch)
            )

        average_training_loss = (
            total_training_loss
            / len(train_values)
        )

        current_validation_errors = (
            reconstruction_errors(
                model,
                validation_values
            )
        )

        current_validation_loss = (
            current_validation_errors.mean()
        )

        if current_validation_loss < best_validation_loss:
            best_validation_loss = current_validation_loss

            best_weights = {
                name: value.detach().cpu().clone()
                for name, value
                in model.state_dict().items()
            }

            waiting_epochs = 0

        else:
            waiting_epochs += 1

        if epoch == 0 or (epoch + 1) % 10 == 0:
            print(
                stream_name,
                "epoch",
                epoch + 1,
                "training loss",
                round(average_training_loss, 6),
                "validation loss",
                round(current_validation_loss, 6)
            )

        if waiting_epochs >= PATIENCE:
            print(stream_name, "stopped at epoch", epoch + 1)
            break

    model.load_state_dict(best_weights)

    validation_errors = reconstruction_errors(
        model,
        validation_values
    )

    test_errors = reconstruction_errors(
        model,
        test_values
    )

    threshold = np.quantile(
        validation_errors,
        STREAM_THRESHOLD_PERCENTILE
    )

    validation_results = validation_data[
        ["user", "day"]
    ].reset_index(drop=True)

    validation_results[
        f"{stream_name}_score"
    ] = validation_errors

    validation_results[
        f"{stream_name}_percentile"
    ] = percentile_scores(
        validation_errors,
        validation_errors
    )

    validation_results[
        f"{stream_name}_flag"
    ] = (
        validation_errors > threshold
    ).astype(int)

    test_results = test_data[
        ["user", "day"]
    ].reset_index(drop=True)

    test_results[
        f"{stream_name}_score"
    ] = test_errors

    test_results[
        f"{stream_name}_percentile"
    ] = percentile_scores(
        validation_errors,
        test_errors
    )

    test_results[
        f"{stream_name}_flag"
    ] = (
        test_errors > threshold
    ).astype(int)

    return {
        "model": model,
        "scaler": scaler,
        "threshold": float(threshold),
        "validation": validation_results,
        "test": test_results
    }


## Training the three independent models

In [18]:
logon_output = train_stream(
    logon_features,
    LOGON_FEATURES,
    "logon"
)

device_output = train_stream(
    device_features,
    DEVICE_FEATURES,
    "device"
)

file_output = train_stream(
    file_features,
    FILE_FEATURES,
    "file"
)

Known malicious training profiles removed: 6

Training logon autoencoder
Training rows: 987015
Validation rows: 205061
Test rows: 201915
logon epoch 1 training loss 0.407567 validation loss 0.134086
logon epoch 10 training loss 0.0405 validation loss 0.035887
logon epoch 20 training loss 0.033684 validation loss 0.030033
logon epoch 30 training loss 0.030718 validation loss 0.026874
logon epoch 40 training loss 0.028996 validation loss 0.02523
logon epoch 50 training loss 0.027596 validation loss 0.023966
Known malicious training profiles removed: 4

Training device autoencoder
Training rows: 141468
Validation rows: 29017
Test rows: 28504
device epoch 1 training loss 0.945709 validation loss 0.769004
device epoch 10 training loss 0.092126 validation loss 0.084367
device epoch 20 training loss 0.049383 validation loss 0.042876
device epoch 30 training loss 0.036072 validation loss 0.03361
device epoch 40 training loss 0.031911 validation loss 0.02866
device epoch 50 training loss 0.0292

## Merge model scores and calibrate the ensemble on validation data

Both candidate rules use the same three stream percentiles. `mean` preserves the original ensemble. `maximum` prevents an extreme file or device score from being diluted by a lower logon score. Missing streams remain missing in both rules.

For each rule, the code tests six documented percentile thresholds. The choice with the highest validation F1 is used. Ties prefer higher recall and then fewer validation alerts.

In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score
)
from sklearn.model_selection import train_test_split


def merge_streams(logon_result, device_result, file_result):
    merged = (
        logon_result
        .merge(device_result, on=["user", "day"], how="outer")
        .merge(file_result, on=["user", "day"], how="outer")
    )

    percentile_columns = [
        "logon_percentile", "device_percentile", "file_percentile"
    ]
    flag_columns = ["logon_flag", "device_flag", "file_flag"]

    merged["available_streams"] = (
        merged[percentile_columns].notna().sum(axis=1)
    )
    merged["mean_ensemble_score"] = (
        merged[percentile_columns].mean(axis=1, skipna=True)
    )
    merged["max_ensemble_score"] = (
        merged[percentile_columns].max(axis=1, skipna=True)
    )
    merged["flagged_streams"] = (
        merged[flag_columns].fillna(0).sum(axis=1).astype(int)
    )

    # Two independent streams constitute high-confidence technical evidence.
    # A one-stream anomaly remains visible as a review item, but is not treated
    # as equivalent to multi-stream agreement.
    merged["technical_anomaly_flag"] = (
        merged["flagged_streams"] >= 1
    ).astype(int)
    merged["multi_stream_flag"] = (
        merged["flagged_streams"] >= 2
    ).astype(int)

    # Kept for compatibility with the original output. It now means the
    # high-confidence two-stream vote, not a maximum-score threshold.
    merged["ensemble_flag"] = merged["multi_stream_flag"]
    merged["ensemble_score"] = merged["mean_ensemble_score"]
    merged["ensemble_method"] = "two_of_three_vote"
    return merged


validation_ensemble = merge_streams(
    logon_output["validation"],
    device_output["validation"],
    file_output["validation"]
)
test_ensemble = merge_streams(
    logon_output["test"],
    device_output["test"],
    file_output["test"]
)

validation_ensemble = validation_ensemble.merge(
    daily_ground_truth,
    on=["user", "day"],
    how="left"
)
validation_ensemble["ground_truth_label"] = (
    validation_ensemble["ground_truth_label"].fillna(0).astype(int)
)

for stream in ["logon", "device", "file"]:
    for dataframe in [validation_ensemble, test_ensemble]:
        dataframe["has_" + stream] = (
            dataframe[stream + "_percentile"].notna().astype(int)
        )
        dataframe[stream + "_percentile_filled"] = (
            dataframe[stream + "_percentile"].fillna(0.0)
        )

META_FEATURES = [
    "logon_percentile_filled",
    "device_percentile_filled",
    "file_percentile_filled",
    "has_logon", "has_device", "has_file",
    "mean_ensemble_score", "max_ensemble_score",
    "flagged_streams", "available_streams"
]

# Validation is divided internally so threshold selection does not evaluate the
# same rows used to fit the classifier. The final test period remains untouched.
fit_rows, threshold_rows = train_test_split(
    validation_ensemble,
    test_size=0.30,
    random_state=SEED,
    stratify=validation_ensemble["ground_truth_label"]
)

malicious_model = LogisticRegression(
    class_weight="balanced",
    penalty="l2",
    max_iter=1000,
    random_state=SEED
)
malicious_model.fit(
    fit_rows[META_FEATURES],
    fit_rows["ground_truth_label"]
)

threshold_probabilities = malicious_model.predict_proba(
    threshold_rows[META_FEATURES]
)[:, 1]

candidate_precision, candidate_recall, candidate_thresholds = (
    precision_recall_curve(
        threshold_rows["ground_truth_label"],
        threshold_probabilities
    )
)
candidate_f1 = (
    2 * candidate_precision[:-1] * candidate_recall[:-1]
    / np.maximum(
        candidate_precision[:-1] + candidate_recall[:-1],
        1e-12
    )
)

best_f1 = candidate_f1.max()
best_indices = np.flatnonzero(np.isclose(candidate_f1, best_f1))

# If F1 ties, use the strictest threshold, which creates fewer alerts.
best_index = int(best_indices[-1])
malicious_threshold = float(candidate_thresholds[best_index])

for dataframe in [validation_ensemble, test_ensemble]:
    dataframe["malicious_probability"] = malicious_model.predict_proba(
        dataframe[META_FEATURES]
    )[:, 1]
    dataframe["malicious_flag"] = (
        dataframe["malicious_probability"] >= malicious_threshold
    ).astype(int)

    dataframe["semantic_category"] = np.select(
        [
            dataframe["malicious_flag"] == 1,
            dataframe["technical_anomaly_flag"] == 1
        ],
        [
            "Suspected Malicious",
            "Unusual-Legitimate Review"
        ],
        default="Ordinary"
    )

calibration_table = pd.DataFrame({
    "malicious_threshold": [malicious_threshold],
    "threshold_precision": [candidate_precision[best_index]],
    "threshold_recall": [candidate_recall[best_index]],
    "threshold_f1": [candidate_f1[best_index]],
    "threshold_rows": [len(threshold_rows)],
    "threshold_malicious_rows": [int(threshold_rows["ground_truth_label"].sum())]
})

# Compatibility variable for the saved threshold table. Technical fusion is a
# vote count, so its high-confidence cutoff is two agreeing streams.
ensemble_threshold = 2.0
selected_ensemble_method = "two_of_three_vote"

print("Technical ensemble:", selected_ensemble_method)
print("Technical high-confidence cutoff: 2 agreeing streams")
print("Malicious probability threshold:", malicious_threshold)
print("Technical review items:", int(test_ensemble["technical_anomaly_flag"].sum()))
print("Multi-stream technical anomalies:", int(test_ensemble["multi_stream_flag"].sum()))
print("Suspected-malicious alerts:", int(test_ensemble["malicious_flag"].sum()))
display(calibration_table)


Technical ensemble: two_of_three_vote
Technical high-confidence cutoff: 2 agreeing streams
Malicious probability threshold: 0.9848469862141195
Technical review items: 13172
Multi-stream technical anomalies: 1081
Suspected-malicious alerts: 919


,malicious_threshold,threshold_precision,threshold_recall,threshold_f1,threshold_rows,threshold_malicious_rows
0,0.984847,0.003623,0.125,0.007042,61519,8


## Creating the technical anomaly categories

In [20]:
def technical_category(row):
    if row["flagged_streams"] == 0:
        return "No Technical Anomaly"

    available = [
        stream for stream in ["logon", "device", "file"]
        if not pd.isna(row[stream + "_percentile"])
    ]

    if row["flagged_streams"] >= 2:
        return "Multi-Stream Anomaly"

    dominant = max(
        available,
        key=lambda stream: row[stream + "_percentile"]
    )
    return dominant.capitalize() + "-Stream Review"


test_ensemble["technical_category"] = test_ensemble.apply(
    technical_category,
    axis=1
)


## Loading and preparing the psychometric file

In [21]:
psychometric = pd.read_csv(
    PSYCHOMETRIC_PATH,
    low_memory=False
)

psychometric.columns = [
    column.strip().lower()
    for column in psychometric.columns
]

if "user_id" in psychometric.columns:
    psychometric = psychometric.rename(
        columns={"user_id": "user"}
    )

elif "userid" in psychometric.columns:
    psychometric = psychometric.rename(
        columns={"userid": "user"}
    )

elif "user" not in psychometric.columns:
    raise ValueError(
        "No user ID column was found in psychometric.csv."
    )

required_ocean_columns = [
    "o",
    "c",
    "e",
    "a",
    "n"
]

missing_ocean_columns = [
    column
    for column in required_ocean_columns
    if column not in psychometric.columns
]

if missing_ocean_columns:
    raise ValueError(
        f"Missing OCEAN columns: {missing_ocean_columns}"
    )

psychometric = psychometric.rename(
    columns={
        "o": "openness",
        "c": "conscientiousness",
        "e": "extraversion",
        "a": "agreeableness",
        "n": "neuroticism"
    }
)

psychometric["user"] = (
    psychometric["user"]
    .astype(str)
    .str.strip()
    .str.upper()
)

OCEAN_TRAITS = [
    "openness",
    "conscientiousness",
    "extraversion",
    "agreeableness",
    "neuroticism"
]

for trait in OCEAN_TRAITS:
    psychometric[trait] = pd.to_numeric(
        psychometric[trait],
        errors="coerce"
    )

psychometric = (
    psychometric
    .dropna(subset=OCEAN_TRAITS)
    .drop_duplicates(subset=["user"])
)

## Calculating the OCEAN bands using training users

In [22]:
training_users = set(
    pd.concat(
        [
            logon_features[
                logon_features["day"] < TRAIN_END_DAY
            ]["user"],

            device_features[
                device_features["day"] < TRAIN_END_DAY
            ]["user"],

            file_features[
                file_features["day"] < TRAIN_END_DAY
            ]["user"]
        ],
        ignore_index=True
    ).unique()
)

psychometric_reference = psychometric[
    psychometric["user"].isin(training_users)
].copy()

if psychometric_reference.empty:
    raise ValueError(
        "No psychometric users matched the activity datasets."
    )

low_cutoffs = psychometric_reference[
    OCEAN_TRAITS
].quantile(OCEAN_LOW_PERCENTILE)

high_cutoffs = psychometric_reference[
    OCEAN_TRAITS
].quantile(OCEAN_HIGH_PERCENTILE)

final_results = test_ensemble.merge(
    psychometric,
    on="user",
    how="left"
)

print(
    "Psychometric match rate:",
    round(
        final_results["openness"].notna().mean() * 100,
        2
    ),
    "%"
)

Psychometric match rate: 100.0 %


## assigning the psychometric categories

In [23]:
def trait_band(value, trait):
    if pd.isna(value):
        return "Unavailable"

    if value <= low_cutoffs[trait]:
        return "Low"

    if value >= high_cutoffs[trait]:
        return "High"

    return "Typical"


for trait in OCEAN_TRAITS:
    final_results[
        f"{trait}_band"
    ] = final_results[trait].apply(
        lambda value, current_trait=trait:
        trait_band(value, current_trait)
    )


trait_names = {
    "openness": "Openness",
    "conscientiousness": "Conscientiousness",
    "extraversion": "Extraversion",
    "agreeableness": "Agreeableness",
    "neuroticism": "Neuroticism"
}


def ocean_context(row):
    if row["semantic_category"] == "Ordinary":
        return "Not Applied to Normal Record"

    if pd.isna(row["openness"]):
        return "Psychometric Data Unavailable"

    labels = []

    for trait in OCEAN_TRAITS:
        band = row[f"{trait}_band"]

        if band == "Low" or band == "High":
            labels.append(
                band + " " + trait_names[trait] + " Profile"
            )

    if not labels:
        return "No Extreme OCEAN Trait"

    return " | ".join(labels)


final_results["ocean_context"] = (
    final_results.apply(
        ocean_context,
        axis=1
    )
)


## Creating the final combined category

In [24]:
def final_category(row):
    if row["semantic_category"] == "Ordinary":
        return "Ordinary"

    return (
        row["semantic_category"]
        + " | "
        + row["technical_category"]
        + " | "
        + row["ocean_context"]
    )


final_results["final_category"] = final_results.apply(
    final_category,
    axis=1
)


## Arranging and Saving the results

In [25]:
output_columns = [
    "user",
    "day",

    "logon_score",
    "logon_percentile",
    "logon_flag",

    "device_score",
    "device_percentile",
    "device_flag",

    "file_score",
    "file_percentile",
    "file_flag",

    "mean_ensemble_score",
    "max_ensemble_score",
    "ensemble_method",
    "ensemble_score",
    "ensemble_flag",
    "technical_anomaly_flag",
    "multi_stream_flag",
    "available_streams",
    "flagged_streams",
    "technical_category",
    "malicious_probability",
    "malicious_flag",
    "semantic_category",

    "openness",
    "openness_band",

    "conscientiousness",
    "conscientiousness_band",

    "extraversion",
    "extraversion_band",

    "agreeableness",
    "agreeableness_band",

    "neuroticism",
    "neuroticism_band",

    "ocean_context",
    "final_category"
]

final_output = final_results[
    output_columns
].sort_values(
    "malicious_probability",
    ascending=False
).reset_index(drop=True)

RESULT_PATH = os.path.join(
    OUTPUT_FOLDER,
    "ensemble_anomaly_results.csv"
)

THRESHOLD_PATH = os.path.join(
    OUTPUT_FOLDER,
    "model_thresholds.csv"
)

final_output.to_csv(
    RESULT_PATH,
    index=False
)

threshold_table = pd.DataFrame(
    {
        "model": [
            "logon",
            "device",
            "file",
            "two_stream_vote_count"
        ],
        "threshold": [
            logon_output["threshold"],
            device_output["threshold"],
            file_output["threshold"],
            ensemble_threshold
        ]
    }
)

calibration_table.to_csv(
    os.path.join(OUTPUT_FOLDER, "ensemble_calibration.csv"),
    index=False
)

threshold_table.to_csv(
    THRESHOLD_PATH,
    index=False
)

torch.save(
    logon_output["model"].state_dict(),
    os.path.join(
        OUTPUT_FOLDER,
        "logon_autoencoder.pth"
    )
)

torch.save(
    device_output["model"].state_dict(),
    os.path.join(
        OUTPUT_FOLDER,
        "device_autoencoder.pth"
    )
)

torch.save(
    file_output["model"].state_dict(),
    os.path.join(
        OUTPUT_FOLDER,
        "file_autoencoder.pth"
    )
)

pd.to_pickle(
    logon_output["scaler"],
    os.path.join(
        OUTPUT_FOLDER,
        "logon_scaler.pkl"
    )
)

pd.to_pickle(
    device_output["scaler"],
    os.path.join(
        OUTPUT_FOLDER,
        "device_scaler.pkl"
    )
)

pd.to_pickle(
    malicious_model,
    os.path.join(OUTPUT_FOLDER, "malicious_logistic_model.pkl")
)

pd.to_pickle(
    file_output["scaler"],
    os.path.join(
        OUTPUT_FOLDER,
        "file_scaler.pkl"
    )
)

print("Results saved to:", RESULT_PATH)
print("Thresholds saved to:", THRESHOLD_PATH)

final_output.head(20)


Results saved to: /content/drive/MyDrive/r6.2/CERT_model_outputs/ensemble_anomaly_results.csv
Thresholds saved to: /content/drive/MyDrive/r6.2/CERT_model_outputs/model_thresholds.csv


,user,day,logon_score,logon_percentile,logon_flag,device_score,device_percentile,device_flag,file_score,file_percentile,...,conscientiousness,conscientiousness_band,extraversion,extraversion_band,agreeableness,agreeableness_band,neuroticism,neuroticism_band,ocean_context,final_category
0,BYF3784,2011-05-06,1.596138,0.999132,1,0.000851,0.000448,0.0,0.037457,0.904444,...,26,Typical,39,Typical,23,Typical,29,Typical,No Extreme OCEAN Trait,Suspected Malicious | Logon-Stream Review | No...
1,SGC2111,2011-04-04,0.232245,0.984497,1,0.003067,0.087811,0.0,0.060649,0.942043,...,35,Typical,31,Typical,44,Typical,36,Typical,No Extreme OCEAN Trait,Suspected Malicious | Logon-Stream Review | No...
2,CAR0992,2011-03-18,0.075343,0.942373,0,0.004902,0.239205,0.0,0.053770,0.933705,...,40,Typical,15,Typical,26,Typical,28,Typical,No Extreme OCEAN Trait,Suspected Malicious | No Technical Anomaly | N...
3,SCM3407,2011-03-29,0.363414,0.991403,1,0.001447,0.007685,0.0,0.037284,0.904083,...,14,Low,35,Typical,44,Typical,36,Typical,Low Conscientiousness Profile,Suspected Malicious | Logon-Stream Review | Lo...
4,UXC1495,2011-03-30,0.399283,0.992319,1,0.004532,0.210532,0.0,0.066913,0.948031,...,47,High,37,Typical,41,Typical,32,Typical,High Conscientiousness Profile,Suspected Malicious | Logon-Stream Review | Hi...
5,RJF3299,2011-03-22,0.058574,0.926139,0,0.004194,0.182996,0.0,0.056014,0.936643,...,14,Low,34,Typical,35,Typical,31,Typical,Low Conscientiousness Profile,Suspected Malicious | No Technical Anomaly | L...
6,NMM1560,2011-04-28,0.438848,0.993202,1,0.002128,0.032085,0.0,0.033862,0.895113,...,36,Typical,21,Typical,42,Typical,36,Typical,No Extreme OCEAN Trait,Suspected Malicious | Logon-Stream Review | No...
7,TZM0851,2011-04-06,0.080891,0.946582,0,0.001686,0.014474,0.0,0.023927,0.857695,...,42,Typical,41,Typical,43,Typical,36,Typical,No Extreme OCEAN Trait,Suspected Malicious | No Technical Anomaly | N...
8,TCB0550,2011-05-04,0.035597,0.879177,0,0.001656,0.013268,0.0,0.058696,0.939897,...,42,Typical,48,High,39,Typical,34,Typical,High Extraversion Profile,Suspected Malicious | No Technical Anomaly | H...
9,DEP0163,2011-04-26,0.064397,0.932713,0,0.001750,0.016887,0.0,0.029660,0.882007,...,19,Typical,47,High,35,Typical,21,Low,High Extraversion Profile | Low Neuroticism Pr...,Suspected Malicious | No Technical Anomaly | H...


# Improvements after the original model

The original model ends above. The cells below evaluate the test results against a verified `ground_truth.csv`. The autoencoders remain unsupervised because these labels are used only after training.

Required columns in `ground_truth.csv`: `user`, `day`, `scenario_id`. Each row must be a verified malicious user-day taken from the answer material for the same CERT release.


In [26]:
# Find ground_truth.csv in the same Drive folder as the CERT files.
GROUND_TRUTH_PATH = os.path.join(
    DATA_FOLDER,
    "ground_truth.csv"
)

if not os.path.exists(GROUND_TRUTH_PATH):
    print(
        "Evaluation skipped: ground_truth.csv was not found in",
        DATA_FOLDER
    )
    print(
        "Create it from the official answer material, then run "
        "the evaluation cells again."
    )
else:
    print("Ground truth found:", GROUND_TRUTH_PATH)


Ground truth found: /content/drive/MyDrive/r6.2/ground_truth.csv


## Load and validate ground truth

Run this cell only after `ground_truth.csv` exists.


In [27]:
ground_truth = pd.read_csv(GROUND_TRUTH_PATH)

required_ground_truth_columns = [
    "user",
    "day",
    "scenario_id"
]

missing_columns = [
    column
    for column in required_ground_truth_columns
    if column not in ground_truth.columns
]

if missing_columns:
    raise ValueError(
        "ground_truth.csv is missing: "
        + str(missing_columns)
    )

ground_truth["user"] = (
    ground_truth["user"]
    .astype(str)
    .str.strip()
    .str.upper()
)

ground_truth["day"] = (
    pd.to_datetime(
        ground_truth["day"],
        errors="coerce"
    )
    .dt.normalize()
)

if ground_truth["day"].isna().any():
    raise ValueError(
        "Some ground-truth dates could not be parsed."
    )

ground_truth = ground_truth.drop_duplicates(
    subset=["user", "day", "scenario_id"]
).reset_index(drop=True)

daily_ground_truth = (
    ground_truth
    .groupby(
        ["user", "day"],
        as_index=False
    )
    .agg(
        ground_truth_label=("scenario_id", "size"),
        scenario_id=(
            "scenario_id",
            lambda values:
            " | ".join(sorted(set(values.astype(str))))
        )
    )
)

daily_ground_truth["ground_truth_label"] = 1

print("Malicious user-days:", len(daily_ground_truth))
print(
    "Malicious users:",
    daily_ground_truth["user"].nunique()
)


Malicious user-days: 44
Malicious users: 5


## Locate the answers archive

In [28]:
import glob
import os

ANSWER_ARCHIVE_PATH = os.path.join(
    DATA_FOLDER,
    "answers.tar.bz2"
)

if not os.path.exists(ANSWER_ARCHIVE_PATH):
    answer_candidates = glob.glob(
        DRIVE_ROOT + "/**/answers.tar.bz2",
        recursive=True
    )

    if len(answer_candidates) == 0:
        raise FileNotFoundError(
            "answers.tar.bz2 was not found in My Drive."
        )

    if len(answer_candidates) > 1:
        print("Multiple answer archives were found:")

        for path in answer_candidates:
            print(path)

        raise ValueError(
            "Set ANSWER_ARCHIVE_PATH manually using "
            "the correct path printed above."
        )

    ANSWER_ARCHIVE_PATH = answer_candidates[0]

print(
    "Answers archive:",
    ANSWER_ARCHIVE_PATH
)

Answers archive: /content/drive/MyDrive/r6.2/answers.tar.bz2


## Creating ground_truth.csv

In [29]:
import csv
import io
import tarfile

CERT_RELEASE = "6.2"

ground_truth_records = []

with tarfile.open(
    ANSWER_ARCHIVE_PATH,
    mode="r:bz2"
) as archive:

    insiders_file = archive.extractfile(
        "answers/insiders.csv"
    )

    if insiders_file is None:
        raise FileNotFoundError(
            "answers/insiders.csv was not found "
            "inside the archive."
        )

    insiders = pd.read_csv(
        insiders_file,
        dtype=str
    )

    release_incidents = insiders[
        insiders["dataset"] == CERT_RELEASE
    ].copy()

    if release_incidents.empty:
        raise ValueError(
            "No answer records were found for CERT "
            + CERT_RELEASE
        )

    print(
        "Incidents found:",
        len(release_incidents)
    )

    for _, incident in release_incidents.iterrows():

        insider_user = (
            incident["user"]
            .strip()
            .upper()
        )

        details_filename = incident["details"]

        details_path = (
            "answers/"
            + details_filename
        )

        details_file = archive.extractfile(
            details_path
        )

        if details_file is None:
            raise FileNotFoundError(
                "Missing answer file: "
                + details_path
            )

        text_file = io.TextIOWrapper(
            details_file,
            encoding="utf-8",
            errors="replace",
            newline=""
        )

        rows = csv.reader(text_file)

        for row in rows:

            if len(row) < 4:
                continue

            event_date = pd.to_datetime(
                row[2],
                format=DATE_FORMAT,
                errors="coerce"
            )

            event_user = (
                row[3]
                .strip()
                .upper()
            )

            if pd.isna(event_date):
                continue

            if event_user != insider_user:
                continue

            ground_truth_records.append(
                {
                    "user": insider_user,
                    "day": event_date.normalize(),
                    "scenario_id": (
                        "r"
                        + CERT_RELEASE
                        + "-scenario-"
                        + str(incident["scenario"])
                    )
                }
            )

Incidents found: 5


## Removing duplicate user-days

In [30]:
ground_truth = pd.DataFrame(
    ground_truth_records
)

if ground_truth.empty:
    raise ValueError(
        "No malicious user-days were extracted."
    )

ground_truth = (
    ground_truth
    .drop_duplicates(
        subset=[
            "user",
            "day",
            "scenario_id"
        ]
    )
    .sort_values(
        [
            "day",
            "user",
            "scenario_id"
        ]
    )
    .reset_index(drop=True)
)

GROUND_TRUTH_PATH = os.path.join(
    DATA_FOLDER,
    "ground_truth.csv"
)

ground_truth.to_csv(
    GROUND_TRUTH_PATH,
    index=False
)

print(
    "Ground truth saved to:",
    GROUND_TRUTH_PATH
)

print(
    "Malicious users:",
    ground_truth["user"].nunique()
)

print(
    "Malicious user-days:",
    len(ground_truth)
)

ground_truth

Ground truth saved to: /content/drive/MyDrive/r6.2/ground_truth.csv
Malicious users: 5
Malicious user-days: 44


,user,day,scenario_id
0,PLJ1771,2010-08-12,r6.2-scenario-3
1,PLJ1771,2010-08-13,r6.2-scenario-3
2,ACM2278,2010-08-18,r6.2-scenario-1
3,ACM2278,2010-08-19,r6.2-scenario-1
4,ACM2278,2010-08-24,r6.2-scenario-1
5,MBG3183,2010-10-12,r6.2-scenario-5
6,CMP2946,2011-02-02,r6.2-scenario-2
7,CMP2946,2011-02-03,r6.2-scenario-2
8,CMP2946,2011-02-04,r6.2-scenario-2
9,CMP2946,2011-02-07,r6.2-scenario-2


## Creating daily ground truth

In [31]:
daily_ground_truth = (
    ground_truth
    .groupby(
        ["user", "day"],
        as_index=False
    )
    .agg(
        scenario_id=(
            "scenario_id",
            lambda values:
            " | ".join(
                sorted(
                    set(
                        values.astype(str)
                    )
                )
            )
        )
    )
)

daily_ground_truth[
    "ground_truth_label"
] = 1

print(
    "Daily ground-truth rows:",
    len(daily_ground_truth)
)

print(
    "Malicious users:",
    daily_ground_truth["user"].nunique()
)

daily_ground_truth.head()

Daily ground-truth rows: 44
Malicious users: 5


,user,day,scenario_id,ground_truth_label
0,ACM2278,2010-08-18,r6.2-scenario-1,1
1,ACM2278,2010-08-19,r6.2-scenario-1,1
2,ACM2278,2010-08-24,r6.2-scenario-1,1
3,CDE1846,2011-02-21,r6.2-scenario-4,1
4,CDE1846,2011-03-17,r6.2-scenario-4,1


## Merge labels and confirm test coverage


In [32]:
evaluation_results = final_output.merge(
    daily_ground_truth,
    on=["user", "day"],
    how="left"
)

evaluation_results["ground_truth_label"] = (
    evaluation_results["ground_truth_label"]
    .fillna(0)
    .astype(int)
)

evaluation_results["scenario_id"] = (
    evaluation_results["scenario_id"]
    .fillna("Background")
)

malicious_test_rows = int(
    evaluation_results["ground_truth_label"].sum()
)

print("Test rows:", len(evaluation_results))
print("Malicious test user-days:", malicious_test_rows)

if malicious_test_rows == 0:
    raise ValueError(
        "The test period contains no verified malicious user-days. "
        "Performance cannot be evaluated with this split."
    )


Test rows: 201915
Malicious test user-days: 12


## Calculate performance metrics


In [33]:
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)


def metric_row(flag_column, score_column, model_name):
    y_true = evaluation_results["ground_truth_label"]
    y_pred = evaluation_results[flag_column].astype(int)
    y_score = evaluation_results[score_column]

    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    return {
        "model": model_name,
        "true_positives": int(tp),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_negatives": int(tn),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "average_precision": average_precision_score(y_true, y_score),
        "alerts": int(y_pred.sum())
    }


metrics_table = pd.DataFrame([
    metric_row(
        "technical_anomaly_flag",
        "mean_ensemble_score",
        "Any-stream technical review"
    ),
    metric_row(
        "multi_stream_flag",
        "mean_ensemble_score",
        "Two-stream technical vote"
    ),
    metric_row(
        "malicious_flag",
        "malicious_probability",
        "Personal hybrid suspected-malicious model"
    )
])

metrics_table


,model,true_positives,false_positives,false_negatives,true_negatives,precision,recall,f1,balanced_accuracy,average_precision,alerts
0,Any-stream technical review,8,13164,4,188739,0.000607,0.666667,0.001214,0.800734,0.000949,13172
1,Two-stream technical vote,7,1074,5,200829,0.006475,0.583333,0.012809,0.789007,0.000949,1081
2,Personal hybrid suspected-malicious model,0,919,12,200984,0.000000,0.000000,0.000000,0.497724,0.000342,919


## Compare individual streams with the ensemble


In [34]:
def evaluate_scores(
    dataframe,
    score_column,
    flag_column,
    model_name
):
    available = dataframe.dropna(
        subset=[score_column, flag_column]
    ).copy()

    true_values = available["ground_truth_label"]
    predictions = available[flag_column].astype(int)
    scores = available[score_column]

    if true_values.nunique() == 2:
        model_ap = average_precision_score(
            true_values,
            scores
        )
    else:
        model_ap = np.nan

    return {
        "model": model_name,
        "rows_evaluated": len(available),
        "malicious_rows": int(true_values.sum()),
        "precision": precision_score(
            true_values,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            true_values,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            true_values,
            predictions,
            zero_division=0
        ),
        "average_precision": model_ap
    }


model_comparison = pd.DataFrame(
    [
        evaluate_scores(
            evaluation_results,
            "logon_percentile",
            "logon_flag",
            "Logon autoencoder"
        ),
        evaluate_scores(
            evaluation_results,
            "device_percentile",
            "device_flag",
            "Device autoencoder"
        ),
        evaluate_scores(
            evaluation_results,
            "file_percentile",
            "file_flag",
            "File autoencoder"
        ),
        evaluate_scores(
            evaluation_results,
            "ensemble_score",
            "ensemble_flag",
            "Two-stream technical vote"
        ),
        evaluate_scores(
            evaluation_results,
            "malicious_probability",
            "malicious_flag",
            "Personal hybrid suspected-malicious model"
        )
    ]
)

model_comparison


,model,rows_evaluated,malicious_rows,precision,recall,f1,average_precision
0,Logon autoencoder,201915,12,0.000654,0.583333,0.001306,0.000478
1,Device autoencoder,28504,4,0.000000,0.000000,0.000000,0.000180
2,File autoencoder,44677,12,0.003426,0.666667,0.006817,0.281538
3,Two-stream technical vote,201915,12,0.006475,0.583333,0.012809,0.000949
4,Personal hybrid suspected-malicious model,201915,12,0.000000,0.000000,0.000000,0.000342


## Additional practical diagnostics

These results explain whether the improvement came from better ranking or merely from issuing fewer alerts. They also report the exact rank of every malicious test user-day.

In [35]:
ranked_results = (
    evaluation_results
    .sort_values("malicious_probability", ascending=False)
    .reset_index(drop=True)
)

ranked_results["rank"] = np.arange(1, len(ranked_results) + 1)

malicious_test_ranks = ranked_results[
    ranked_results["ground_truth_label"] == 1
][
    [
        "user",
        "day",
        "scenario_id",
        "rank",
        "ensemble_score",
        "ensemble_flag",
        "malicious_probability",
        "malicious_flag",
        "semantic_category"
    ]
]

stream_anomaly_counts = pd.DataFrame({
    "model": ["Logon", "Device", "File", "Any-stream review", "Two-stream vote", "Suspected malicious"],
    "detected_anomalies": [
        int(evaluation_results["logon_flag"].fillna(0).sum()),
        int(evaluation_results["device_flag"].fillna(0).sum()),
        int(evaluation_results["file_flag"].fillna(0).sum()),
        int(evaluation_results["technical_anomaly_flag"].sum()),
        int(evaluation_results["multi_stream_flag"].sum()),
        int(evaluation_results["malicious_flag"].sum())
    ]
})

print("Anomalies detected by each model:")
display(stream_anomaly_counts)

print("Ranks of the verified malicious test user-days:")
display(malicious_test_ranks)


Anomalies detected by each model:


,model,detected_anomalies
0,Logon,10705
1,Device,1410
2,File,2335
3,Any-stream review,13172
4,Two-stream vote,1081
5,Suspected malicious,919


Ranks of the verified malicious test user-days:


,user,day,scenario_id,rank,ensemble_score,ensemble_flag,malicious_probability,malicious_flag,semantic_category
2007,CMP2946,2011-03-30,r6.2-scenario-2,2008,0.575089,0,0.979616,0,Ordinary
9693,CMP2946,2011-03-22,r6.2-scenario-2,9694,0.621538,0,0.896693,0,Ordinary
10302,CMP2946,2011-03-23,r6.2-scenario-2,10303,0.726220,0,0.883443,0,Ordinary
11495,CMP2946,2011-03-16,r6.2-scenario-2,11496,0.740168,0,0.853944,0,Ordinary
24259,CDE1846,2011-03-17,r6.2-scenario-4,24260,0.964376,0,0.034566,0,Unusual-Legitimate Review
24484,CDE1846,2011-04-06,r6.2-scenario-4,24485,0.991232,1,0.026930,0,Unusual-Legitimate Review
24574,CDE1846,2011-04-25,r6.2-scenario-4,24575,0.983112,1,0.024779,0,Unusual-Legitimate Review
24631,CDE1846,2011-03-22,r6.2-scenario-4,24632,0.979934,1,0.023960,0,Unusual-Legitimate Review
24688,CDE1846,2011-04-15,r6.2-scenario-4,24689,0.976151,1,0.023069,0,Unusual-Legitimate Review
24717,CDE1846,2011-04-11,r6.2-scenario-4,24718,0.974842,1,0.022677,0,Unusual-Legitimate Review
